In [1]:
import os
import pandas as pd
import numpy as np
import joblib
import json

from sklearn.ensemble import RandomForestClassifier

# ===============================
# BASE DIR
# ===============================
BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))

PROFILE_PATH = os.path.join(BASE_DIR, "reports", "profiling_reports", "dataset_profile.csv")
CLEAN_PATH = os.path.join(BASE_DIR, "data", "clean", "clean_dataset.csv")
TRAINING_PATH = os.path.join(BASE_DIR, "models", "semantic_training_data.csv")
MODEL_PATH = os.path.join(BASE_DIR, "models", "semantic_model.pkl")
REPORT_PATH = os.path.join(BASE_DIR, "reports", "semantic_reports", "semantic_model_report.json")

In [2]:
if not os.path.exists(PROFILE_PATH):
    raise FileNotFoundError("dataset_profile not found")

profile = pd.read_csv(PROFILE_PATH)

if profile.empty:
    raise ValueError("dataset_profile is empty")

df_clean = None
if os.path.exists(CLEAN_PATH):
    df_clean = pd.read_csv(CLEAN_PATH)

existing_training = None
if os.path.exists(TRAINING_PATH):
    existing_training = pd.read_csv(TRAINING_PATH)

print("✅ Inputs loaded")

✅ Inputs loaded


In [3]:
features = []

for col in profile["column_name"]:
    col_data = None
    
    if df_clean is not None and col in df_clean.columns:
        col_data = df_clean[col]
    else:
        continue  # skip if no data

    series = col_data.astype(str)

    total = len(series)

    numeric_ratio = series.str.match(r'^-?\d+(\.\d+)?$').mean()
    alpha_ratio = series.str.match(r'^[A-Za-z]+$').mean()
    missing_ratio = series.isin(["", "?", "nan", "None"]).mean()
    avg_length = series.str.len().mean()
    unique_count = series.nunique()

    has_email_pattern = series.str.contains("@").any()
    has_date_pattern = series.str.contains(r'\d{4}-\d{2}-\d{2}').any()

    features.append({
        "column_name": col,
        "numeric_ratio": numeric_ratio,
        "alpha_ratio": alpha_ratio,
        "missing_ratio": missing_ratio,
        "avg_length": avg_length,
        "unique_count": unique_count,
        "has_email": int(has_email_pattern),
        "has_date": int(has_date_pattern)
    })

feature_df = pd.DataFrame(features)

if feature_df.empty:
    raise ValueError("Feature dataset is empty")

print("✅ Features built")

✅ Features built


In [4]:
# ============================================================
# SAFE + SEMI-AUTOMATIC LABELING
# ============================================================

def infer_label(row):
    col = row["column_name"]
    col_lower = col.lower()

    numeric_ratio = row["numeric_ratio"]
    alpha_ratio = row["alpha_ratio"]
    unique_count = row["unique_count"]
    has_email = row["has_email"]
    has_date = row["has_date"]

    # -------------------------------
    # 1. Identifier (STRICT)
    # -------------------------------
    if "id" in col_lower:
        return "identifier"

    # -------------------------------
    # 2. Target column (dataset-specific but safe)
    # -------------------------------
    if col_lower in ["income", "target", "label"]:
        return "target"

    # -------------------------------
    # 3. Email detection
    # -------------------------------
    if has_email == 1:
        return "email"

    # -------------------------------
    # 4. Date detection
    # -------------------------------
    if has_date == 1:
        return "date"

    # -------------------------------
    # 5. Numeric detection
    # -------------------------------
    if numeric_ratio > 0.95:
        return "numeric"

    # -------------------------------
    # 6. Categorical detection
    # -------------------------------
    if alpha_ratio > 0.5 and unique_count < 100:
        return "categorical"

    # -------------------------------
    # 7. Fallback (DO NOT GUESS)
    # -------------------------------
    return None


# Apply inference
feature_df["label"] = feature_df.apply(infer_label, axis=1)

# Keep only confident labels
labeled_df = feature_df.dropna(subset=["label"])

if labeled_df.empty:
    raise ValueError("No labeled data available")

print("✅ Labels assigned (rule-based)")
print(labeled_df[["column_name", "label"]])

✅ Labels assigned (rule-based)
  column_name        label
0          id   identifier
1         age      numeric
2      gender  categorical


In [5]:
# ============================================================
# INCREMENTAL TRAINING DATASET
# ============================================================

# Keep only required columns
training_cols = [
    "column_name",
    "numeric_ratio",
    "alpha_ratio",
    "missing_ratio",
    "avg_length",
    "unique_count",
    "has_email",
    "has_date",
    "label"
]

new_training = labeled_df[training_cols].copy()

# Load existing if present
if os.path.exists(TRAINING_PATH):
    existing_training = pd.read_csv(TRAINING_PATH)

    if not existing_training.empty:
        combined = pd.concat([existing_training, new_training], ignore_index=True)
    else:
        combined = new_training.copy()
else:
    combined = new_training.copy()

# Remove duplicates deterministically
combined = combined.drop_duplicates(subset=["column_name"], keep="last")

if combined.empty:
    raise ValueError("Training dataset empty after merge")

# Save updated dataset
os.makedirs(os.path.dirname(TRAINING_PATH), exist_ok=True)
combined.to_csv(TRAINING_PATH, index=False)

print("✅ Training dataset updated")
print("Total samples:", len(combined))

✅ Training dataset updated
Total samples: 6


In [6]:
# ============================================================
# TRAIN MODEL (WITH SAFE IMPUTATION)
# ============================================================

df_train = pd.read_csv(TRAINING_PATH)

if df_train.empty:
    raise ValueError("Training data is empty")

X = df_train.drop(columns=["column_name", "label"])
y = df_train["label"]

# ------------------------------------------------------------
# FIX NULL VALUES (DETERMINISTIC)
# ------------------------------------------------------------

# Numeric features → fill with 0
numeric_cols = X.select_dtypes(include=["float64", "int64"]).columns
X[numeric_cols] = X[numeric_cols].fillna(0)

# Boolean-like features → fill with 0
for col in ["has_email", "has_date"]:
    if col in X.columns:
        X[col] = X[col].fillna(0)

# Final check
if X.isnull().any().any():
    raise ValueError("Null values still present after imputation")

# ------------------------------------------------------------
# TRAIN MODEL
# ------------------------------------------------------------

model = RandomForestClassifier(
    n_estimators=50,
    random_state=42
)

model.fit(X, y)

print("✅ Model trained")
print("Classes learned:", sorted(y.unique()))

✅ Model trained
Classes learned: ['categorical', 'email', 'identifier', 'name', 'numeric', 'salary']


In [7]:
# ============================================================
# SAVE MODEL
# ============================================================

model_bundle = {
    "model": model,
    "features": list(X.columns),
    "classes": sorted(y.unique())
}

os.makedirs(os.path.dirname(MODEL_PATH), exist_ok=True)

joblib.dump(model_bundle, MODEL_PATH)

print(f"✅ Model saved at {MODEL_PATH}")

✅ Model saved at /media/ausaafkhan/New Volume1/project_exhibition/broken_dataset_repair/models/semantic_model.pkl


In [8]:
# ============================================================
# TRAINING REPORT
# ============================================================

report = {
    "num_samples": int(len(y)),
    "num_features": int(X.shape[1]),
    "features": list(X.columns),
    "labels": sorted(y.unique())
}

os.makedirs(os.path.dirname(REPORT_PATH), exist_ok=True)

with open(REPORT_PATH, "w") as f:
    json.dump(report, f, indent=4)

print("✅ Training report saved")
print(report)

✅ Training report saved
{'num_samples': 6, 'num_features': 12, 'features': ['pct_numeric', 'pct_alpha', 'pct_missing', 'avg_length', 'nunique', 'has_email', 'has_date', 'has_numeric_range', 'numeric_ratio', 'alpha_ratio', 'missing_ratio', 'unique_count'], 'labels': ['categorical', 'email', 'identifier', 'name', 'numeric', 'salary']}


In [9]:
# ============================================================
# SANITY CHECK
# ============================================================

loaded = joblib.load(MODEL_PATH)

model_loaded = loaded["model"]

sample = X.iloc[:5]

preds = model_loaded.predict(sample)

print("\nSample predictions:")
print(preds)


Sample predictions:
['name' 'salary' 'email' 'identifier' 'numeric']


In [10]:
# ============================================================
# DEBUG: COLUMN-WISE PREDICTIONS
# ============================================================

debug_df = df_train.copy()

debug_X = debug_df.drop(columns=["column_name", "label"])

# apply same imputation
numeric_cols = debug_X.select_dtypes(include=["float64", "int64"]).columns
debug_X[numeric_cols] = debug_X[numeric_cols].fillna(0)

for col in ["has_email", "has_date"]:
    if col in debug_X.columns:
        debug_X[col] = debug_X[col].fillna(0)

preds = model.predict(debug_X)

debug_df["predicted_label"] = preds

print(debug_df[["column_name", "label", "predicted_label"]])

  column_name        label predicted_label
0        name         name            name
1      salary       salary          salary
2       email        email           email
3          id   identifier      identifier
4         age      numeric         numeric
5      gender  categorical     categorical
